# 10 — Temporal Model Tuning

This notebook performs limited hyperparameter tuning for the two supervised models that demonstrated predictive ability during the baseline experiments:

- Logistic Regression;
- Random Forest.

Model selection is based exclusively on pre-2021 observations using expanding-window temporal validation:

- Train on 2018 → validate on 2019;
- Train on 2018–2019 → validate on 2020.

Hyperparameters are selected using mean Average Precision across the temporal validation folds.

Decision thresholds are subsequently selected using the combined out-of-fold predictions from 2019 and 2020.

After model and threshold selection, the final classifier is trained using 2018–2020 and evaluated on 2021.

The 2021 period is not used for hyperparameter or threshold selection.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

INPUT_PATH = Path(
    "../data/processed/recife_2018_2021_model_dataset.csv"
)

OUTPUT_PATH = Path(
    "../data/processed/tuned_model_results.csv"
)

RANDOM_STATE = 42

TEST_START = pd.Timestamp(
    "2021-01-01 00:00",
    tz="UTC",
)

FORECAST_HORIZONS = [
    1,
    2,
    3,
    4,
    5,
    6,
]


# ============================================================
# 2. FEATURE SET
# ============================================================

FEATURE_COLUMNS = [
    "precipitation_mm",
    "temperature_c",
    "humidity_pct",
    "pressure_station_mb",

    "hour_sin",
    "hour_cos",
    "day_of_year_sin",
    "day_of_year_cos",

    "precipitation_lag_1h",
    "precipitation_lag_2h",
    "precipitation_lag_3h",
    "precipitation_lag_6h",
    "precipitation_sum_3h",
    "precipitation_sum_6h",

    "temperature_c_lag_1h",
    "humidity_pct_lag_1h",
    "pressure_station_mb_lag_1h",

    "temperature_change_1h",
    "humidity_change_1h",
    "pressure_change_1h",
]


# ============================================================
# 3. LOAD DATA
# ============================================================

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {INPUT_PATH}"
    )


df = pd.read_csv(
    INPUT_PATH,
    parse_dates=["datetime_utc"],
)

df = (
    df
    .set_index("datetime_utc")
    .sort_index()
)


print("Dataset loaded.")
print("Rows:", len(df))


# ============================================================
# 4. HYPERPARAMETER GRIDS
# ============================================================

logistic_grid = [
    {
        "C": 0.01,
    },
    {
        "C": 0.1,
    },
    {
        "C": 1.0,
    },
    {
        "C": 10.0,
    },
]


random_forest_grid = []

for max_depth in [
    4,
    8,
    None,
]:

    for min_samples_leaf in [
        1,
        10,
    ]:

        random_forest_grid.append(
            {
                "max_depth":
                    max_depth,

                "min_samples_leaf":
                    min_samples_leaf,
            }
        )


parameter_grids = {
    "logistic_regression":
        logistic_grid,

    "random_forest":
        random_forest_grid,
}


# ============================================================
# 5. MODEL FACTORY
# ============================================================

def build_model(
    model_name,
    parameters,
):

    if model_name == "logistic_regression":

        return Pipeline(
            steps=[
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "classifier",
                    LogisticRegression(
                        C=parameters["C"],
                        class_weight="balanced",
                        max_iter=3000,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        )


    if model_name == "random_forest":

        return RandomForestClassifier(
            n_estimators=300,

            max_depth=parameters[
                "max_depth"
            ],

            min_samples_leaf=parameters[
                "min_samples_leaf"
            ],

            class_weight=
                "balanced_subsample",

            random_state=
                RANDOM_STATE,

            n_jobs=-1,
        )


    raise ValueError(
        f"Unknown model: {model_name}"
    )


# ============================================================
# 6. TEMPORAL VALIDATION FOLDS
# ============================================================

temporal_folds = [

    {
        "name": "2018_to_2019",

        "train_end":
            pd.Timestamp(
                "2019-01-01 00:00",
                tz="UTC",
            ),

        "validation_start":
            pd.Timestamp(
                "2019-01-01 00:00",
                tz="UTC",
            ),

        "validation_end":
            pd.Timestamp(
                "2020-01-01 00:00",
                tz="UTC",
            ),
    },

    {
        "name": "2018_2019_to_2020",

        "train_end":
            pd.Timestamp(
                "2020-01-01 00:00",
                tz="UTC",
            ),

        "validation_start":
            pd.Timestamp(
                "2020-01-01 00:00",
                tz="UTC",
            ),

        "validation_end":
            pd.Timestamp(
                "2021-01-01 00:00",
                tz="UTC",
            ),
    },
]


# ============================================================
# 7. TUNING
# ============================================================

final_results = []
selected_models = []


for horizon in FORECAST_HORIZONS:

    print()
    print("=" * 70)
    print(
        f"TUNING HORIZON +{horizon}h"
    )
    print("=" * 70)


    target_column = (
        f"target_{horizon}h"
    )

    eligibility_column = (
        f"eligible_final_{horizon}h"
    )


    eligible = df[
        df[eligibility_column]
    ].copy()


    for model_name, parameter_grid \
            in parameter_grids.items():

        configuration_results = []


        # ====================================================
        # A. TEST EACH PARAMETER CONFIGURATION
        # ====================================================

        for parameters in parameter_grid:

            fold_scores = []


            for fold in temporal_folds:

                train_data = eligible[
                    eligible.index
                    < fold["train_end"]
                ]


                validation_data = eligible[
                    (
                        eligible.index
                        >= fold[
                            "validation_start"
                        ]
                    )
                    &
                    (
                        eligible.index
                        < fold[
                            "validation_end"
                        ]
                    )
                ]


                X_train = train_data[
                    FEATURE_COLUMNS
                ]

                y_train = (
                    train_data[
                        target_column
                    ]
                    .astype(int)
                )


                X_validation = (
                    validation_data[
                        FEATURE_COLUMNS
                    ]
                )

                y_validation = (
                    validation_data[
                        target_column
                    ]
                    .astype(int)
                )


                model = build_model(
                    model_name,
                    parameters,
                )


                model.fit(
                    X_train,
                    y_train,
                )


                validation_scores = (
                    model.predict_proba(
                        X_validation
                    )[:, 1]
                )


                ap = (
                    average_precision_score(
                        y_validation,
                        validation_scores,
                    )
                )


                fold_scores.append(
                    ap
                )


            mean_ap = float(
                np.mean(
                    fold_scores
                )
            )


            configuration_results.append(
                {
                    "parameters":
                        parameters,

                    "mean_average_precision":
                        mean_ap,

                    "fold_scores":
                        fold_scores,
                }
            )


        # ====================================================
        # B. SELECT BEST PARAMETERS
        # ====================================================

        configuration_results = sorted(
            configuration_results,
            key=lambda row:
                row[
                    "mean_average_precision"
                ],
            reverse=True,
        )


        best_configuration = (
            configuration_results[0]
        )


        best_parameters = (
            best_configuration[
                "parameters"
            ]
        )


        print()
        print(model_name)

        print(
            "Best parameters:",
            best_parameters
        )

        print(
            "Mean validation PR-AUC:",
            round(
                best_configuration[
                    "mean_average_precision"
                ],
                3,
            )
        )


        # ====================================================
        # C. GENERATE OUT-OF-FOLD SCORES
        # ====================================================

        oof_true = []

        oof_scores = []


        for fold in temporal_folds:

            train_data = eligible[
                eligible.index
                < fold["train_end"]
            ]


            validation_data = eligible[
                (
                    eligible.index
                    >= fold[
                        "validation_start"
                    ]
                )
                &
                (
                    eligible.index
                    < fold[
                        "validation_end"
                    ]
                )
            ]


            X_train = train_data[
                FEATURE_COLUMNS
            ]

            y_train = (
                train_data[
                    target_column
                ]
                .astype(int)
            )


            X_validation = (
                validation_data[
                    FEATURE_COLUMNS
                ]
            )

            y_validation = (
                validation_data[
                    target_column
                ]
                .astype(int)
            )


            model = build_model(
                model_name,
                best_parameters,
            )


            model.fit(
                X_train,
                y_train,
            )


            scores = (
                model.predict_proba(
                    X_validation
                )[:, 1]
            )


            oof_true.extend(
                y_validation.tolist()
            )

            oof_scores.extend(
                scores.tolist()
            )


        oof_true = np.array(
            oof_true
        )

        oof_scores = np.array(
            oof_scores
        )


        # ====================================================
        # D. SELECT THRESHOLD USING OOF 2019–2020
        # ====================================================

        threshold_rows = []


        for threshold in np.arange(
            0.01,
            1.00,
            0.01,
        ):

            predictions = (
                oof_scores
                >= threshold
            ).astype(int)


            precision = precision_score(
                oof_true,
                predictions,
                zero_division=0,
            )

            recall = recall_score(
                oof_true,
                predictions,
                zero_division=0,
            )

            f1 = f1_score(
                oof_true,
                predictions,
                zero_division=0,
            )


            threshold_rows.append(
                {
                    "threshold":
                        threshold,

                    "precision":
                        precision,

                    "recall":
                        recall,

                    "f1":
                        f1,
                }
            )


        threshold_df = pd.DataFrame(
            threshold_rows
        )


        best_threshold_row = (
            threshold_df
            .sort_values(
                [
                    "f1",
                    "recall",
                    "precision",
                ],
                ascending=False,
            )
            .iloc[0]
        )


        selected_threshold = float(
            best_threshold_row[
                "threshold"
            ]
        )


        print(
            "Selected OOF threshold:",
            round(
                selected_threshold,
                3,
            )
        )

        print(
            "OOF F1:",
            round(
                best_threshold_row[
                    "f1"
                ],
                3,
            )
        )


        # ====================================================
        # E. FINAL TRAINING — 2018–2020
        # ====================================================

        final_train = eligible[
            eligible.index
            < TEST_START
        ]

        test_data = eligible[
            eligible.index
            >= TEST_START
        ]


        X_final_train = (
            final_train[
                FEATURE_COLUMNS
            ]
        )

        y_final_train = (
            final_train[
                target_column
            ]
            .astype(int)
        )


        X_test = test_data[
            FEATURE_COLUMNS
        ]

        y_test = (
            test_data[
                target_column
            ]
            .astype(int)
        )


        final_model = build_model(
            model_name,
            best_parameters,
        )


        final_model.fit(
            X_final_train,
            y_final_train,
        )


        test_scores = (
            final_model.predict_proba(
                X_test
            )[:, 1]
        )


        test_predictions = (
            test_scores
            >= selected_threshold
        ).astype(int)


        # ====================================================
        # F. TEST METRICS
        # ====================================================

        precision = precision_score(
            y_test,
            test_predictions,
            zero_division=0,
        )

        recall = recall_score(
            y_test,
            test_predictions,
            zero_division=0,
        )

        f1 = f1_score(
            y_test,
            test_predictions,
            zero_division=0,
        )


        average_precision = (
            average_precision_score(
                y_test,
                test_scores,
            )
        )


        roc_auc = roc_auc_score(
            y_test,
            test_scores,
        )


        tn, fp, fn, tp = (
            confusion_matrix(
                y_test,
                test_predictions,
                labels=[0, 1],
            )
            .ravel()
        )


        # Critical Success Index
        csi_denominator = (
            tp
            + fp
            + fn
        )

        csi = (
            tp
            / csi_denominator
            if csi_denominator > 0
            else 0
        )


        final_results.append(
            {
                "horizon_hours":
                    horizon,

                "model":
                    model_name,

                "parameters":
                    json.dumps(
                        best_parameters
                    ),

                "threshold":
                    selected_threshold,

                "precision":
                    precision,

                "recall":
                    recall,

                "f1":
                    f1,

                "csi":
                    csi,

                "average_precision":
                    average_precision,

                "roc_auc":
                    roc_auc,

                "true_positives":
                    int(tp),

                "false_positives":
                    int(fp),

                "false_negatives":
                    int(fn),

                "true_negatives":
                    int(tn),
            }
        )


        selected_models.append(
            {
                "horizon_hours":
                    horizon,

                "model":
                    model_name,

                "parameters":
                    json.dumps(
                        best_parameters
                    ),

                "validation_mean_ap":
                    best_configuration[
                        "mean_average_precision"
                    ],

                "threshold":
                    selected_threshold,

                "oof_precision":
                    best_threshold_row[
                        "precision"
                    ],

                "oof_recall":
                    best_threshold_row[
                        "recall"
                    ],

                "oof_f1":
                    best_threshold_row[
                        "f1"
                    ],
            }
        )


        print()
        print("2021 TEST")

        print(
            f"Precision: {precision:.3f}"
        )

        print(
            f"Recall:    {recall:.3f}"
        )

        print(
            f"F1:        {f1:.3f}"
        )

        print(
            f"CSI:       {csi:.3f}"
        )

        print(
            f"PR-AUC:    {average_precision:.3f}"
        )

        print(
            f"ROC-AUC:   {roc_auc:.3f}"
        )


# ============================================================
# 8. RESULTS
# ============================================================

results_df = pd.DataFrame(
    final_results
)

selected_models_df = pd.DataFrame(
    selected_models
)


print()
print(
    "=== SELECTED MODEL CONFIGURATIONS ==="
)

display(
    selected_models_df.round(3)
)


print()
print(
    "=== TUNED 2021 TEST RESULTS ==="
)

display(
    results_df[
        [
            "horizon_hours",
            "model",
            "threshold",
            "precision",
            "recall",
            "f1",
            "csi",
            "average_precision",
            "roc_auc",
            "true_positives",
            "false_positives",
            "false_negatives",
        ]
    ].round(3)
)


# ============================================================
# 9. BEST F1 PER HORIZON
# ============================================================

best_by_horizon = (
    results_df
    .sort_values(
        [
            "horizon_hours",
            "f1",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "horizon_hours",
        as_index=False,
    )
    .first()
)


print()
print(
    "=== BEST MODEL BY HORIZON ==="
)

display(
    best_by_horizon[
        [
            "horizon_hours",
            "model",
            "precision",
            "recall",
            "f1",
            "csi",
            "average_precision",
            "roc_auc",
        ]
    ].round(3)
)


# ============================================================
# 10. SAVE
# ============================================================

results_df.to_csv(
    OUTPUT_PATH,
    index=False,
)


print()
print(
    "Tuned results saved to:"
)

print(
    OUTPUT_PATH
)

Dataset loaded.
Rows: 33871

TUNING HORIZON +1h

logistic_regression
Best parameters: {'C': 0.01}
Mean validation PR-AUC: 0.07
Selected OOF threshold: 0.99
OOF F1: 0.144

2021 TEST
Precision: 0.170
Recall:    0.447
F1:        0.246
CSI:       0.140
PR-AUC:    0.225
ROC-AUC:   0.945

random_forest
Best parameters: {'max_depth': None, 'min_samples_leaf': 10}
Mean validation PR-AUC: 0.023
Selected OOF threshold: 0.18
OOF F1: 0.051

2021 TEST
Precision: 0.100
Recall:    0.421
F1:        0.162
CSI:       0.088
PR-AUC:    0.144
ROC-AUC:   0.932

TUNING HORIZON +2h

logistic_regression
Best parameters: {'C': 0.01}
Mean validation PR-AUC: 0.012
Selected OOF threshold: 0.87
OOF F1: 0.03

2021 TEST
Precision: 0.049
Recall:    0.711
F1:        0.092
CSI:       0.048
PR-AUC:    0.096
ROC-AUC:   0.907

random_forest
Best parameters: {'max_depth': None, 'min_samples_leaf': 10}
Mean validation PR-AUC: 0.015
Selected OOF threshold: 0.13
OOF F1: 0.029

2021 TEST
Precision: 0.056
Recall:    0.395
F1:   

,horizon_hours,model,parameters,validation_mean_ap,threshold,oof_precision,oof_recall,oof_f1
0,1,logistic_regression,"{""C"": 0.01}",0.070,0.99,0.100,0.259,0.144
1,1,random_forest,"{""max_depth"": null, ""min_samples_leaf"": 10}",0.023,0.18,0.030,0.167,0.051
2,2,logistic_regression,"{""C"": 0.01}",0.012,0.87,0.017,0.130,0.030
3,2,random_forest,"{""max_depth"": null, ""min_samples_leaf"": 10}",0.015,0.13,0.016,0.148,0.029
4,3,logistic_regression,"{""C"": 0.01}",0.013,0.95,0.027,0.074,0.040
5,3,random_forest,"{""max_depth"": 4, ""min_samples_leaf"": 1}",0.014,0.77,0.043,0.056,0.049
6,4,logistic_regression,"{""C"": 0.01}",0.016,0.94,0.029,0.093,0.044
7,4,random_forest,"{""max_depth"": 4, ""min_samples_leaf"": 10}",0.015,0.77,0.045,0.056,0.050
8,5,logistic_regression,"{""C"": 0.01}",0.015,0.85,0.021,0.241,0.039
9,5,random_forest,"{""max_depth"": 8, ""min_samples_leaf"": 1}",0.012,0.64,0.028,0.056,0.037



=== TUNED 2021 TEST RESULTS ===


,horizon_hours,model,threshold,precision,recall,f1,csi,average_precision,roc_auc,true_positives,false_positives,false_negatives
0,1,logistic_regression,0.99,0.170,0.447,0.246,0.140,0.225,0.945,17,83,21
1,1,random_forest,0.18,0.100,0.421,0.162,0.088,0.144,0.932,16,144,22
2,2,logistic_regression,0.87,0.049,0.711,0.092,0.048,0.096,0.907,27,520,11
3,2,random_forest,0.13,0.056,0.395,0.098,0.051,0.076,0.866,15,254,23
4,3,logistic_regression,0.95,0.091,0.368,0.146,0.079,0.077,0.904,14,140,24
5,3,random_forest,0.77,0.090,0.526,0.154,0.083,0.071,0.898,20,202,18
6,4,logistic_regression,0.94,0.097,0.368,0.153,0.083,0.075,0.897,14,131,24
7,4,random_forest,0.77,0.064,0.342,0.108,0.057,0.057,0.876,13,189,25
8,5,logistic_regression,0.85,0.038,0.526,0.071,0.037,0.049,0.861,20,504,18
9,5,random_forest,0.64,0.027,0.132,0.045,0.023,0.025,0.823,5,179,33



=== BEST MODEL BY HORIZON ===


,horizon_hours,model,precision,recall,f1,csi,average_precision,roc_auc
0,1,logistic_regression,0.170,0.447,0.246,0.140,0.225,0.945
1,2,random_forest,0.056,0.395,0.098,0.051,0.076,0.866
2,3,random_forest,0.090,0.526,0.154,0.083,0.071,0.898
3,4,logistic_regression,0.097,0.368,0.153,0.083,0.075,0.897
4,5,logistic_regression,0.038,0.526,0.071,0.037,0.049,0.861
5,6,random_forest,0.032,0.237,0.056,0.029,0.022,0.821



Tuned results saved to:
..\data\processed\tuned_model_results.csv


## Conclusion

Temporal hyperparameter tuning confirmed that severe-rainfall predictability is strongest at short forecasting horizons.

The best overall supervised result was obtained by Logistic Regression for the +1h horizon. Using a regularization parameter of `C = 0.01` and a decision threshold of `0.99`, the model achieved:

- Precision: 0.170
- Recall: 0.447
- F1-score: 0.246
- Critical Success Index (CSI): 0.140
- Average Precision (PR-AUC): 0.225
- ROC-AUC: 0.945

The model correctly detected 17 of the 38 severe-rainfall observations in the unseen 2021 test period.

Performance generally deteriorated for longer forecasting horizons, confirming that predictions several hours in advance are substantially more difficult when using observations from a single surface meteorological station.

Different algorithms produced the highest F1-score at different horizons. Logistic Regression performed best at +1h, +4h and +5h, while Random Forest obtained slightly higher F1-scores at +2h, +3h and +6h.

All hyperparameter and decision-threshold selection was performed using observations prior to 2021. The 2021 period remained isolated for final evaluation.

These results complete the supervised-learning stage of the project.